# IMPORTS

In [1]:
import CL_inference as cl_inference
N_threads = cl_inference.train_tools.set_N_threads_(N_threads=1)

import os, sys
import torch
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import CL_inference as cl_inference

%load_ext autoreload

%matplotlib notebook
plt.style.use('default')
plt.close('all')

font, rcnew = cl_inference.plot_utils.matplotlib_default_config()
mpl.rc('font', **font)
plt.rcParams.update(rcnew)
plt.style.use('tableau-colorblind10')
%config InlineBackend.figure_format = 'retina'

device = cl_inference.train_tools.set_torch_device_()

N_threads: 1
Device: cuda


# LOAD DATSETS

In [2]:
path_save = "/cosmos_storage/home/dlopez/Projects/CL_inference/models/join_inference_CL_VICReg_models_illustris_eagle_kmax_0.6/manual-sweep-0"

In [3]:
path_load               = "/cosmos_storage/home/dlopez/Projects/CL_inference/DATASETS/DATASET_kmax_0.6/"
list_model_names        = ["Model_fixed_illustris", "Model_fixed_eagle"]
normalize               = True

NN_augs_batch           = 2
add_noise_Pk            = "cosmic_var_gauss"
kmax                    = 0.6
include_baryon_params   = True

In [4]:
dset = cl_inference.data_tools.def_data_loader(
    path_load               = os.path.join(path_load, "VAL"),
    list_model_names        = list_model_names,
    normalize               = normalize,
    path_save_norm          = None,
    path_load_norm          = path_save,
    NN_augs_batch           = NN_augs_batch,
    add_noise_Pk            = add_noise_Pk,
    kmax                    = kmax,
    include_baryon_params   = include_baryon_params
)

# MODEL ARCHITECTURE

In [5]:
train_mode              = "train_CL_and_inference"
inference_loss          = "MultivariateNormal"

load_encoder_model_path = "None"
input_encoder           = 99
hidden_layers_encoder   = [100, 100]
output_encoder          = 64

hidden_layers_projector = [64, 64, 64, 64]
output_projector        = 64

hidden_layers_inference = [64, 64]
NN_params_out           = 12

load_inference_model_path = "None"
load_projector_model_path = "None"

In [6]:
# ----------------------- define model encoder ----------------------- #

assert input_encoder == dset.xx.shape[-1], "input_encoder from config file must coincide with xx size"

if train_mode == "train_inference_fully_supervised":
    model_encoder = cl_inference.nn_tools.define_MLP_model(
        hidden_layers_encoder+[output_encoder], input_encoder, bn=True, last_bias=True
    ).to(device)
else:
    model_encoder = cl_inference.nn_tools.define_MLP_model(
        hidden_layers_encoder+[output_encoder], input_encoder, bn=True
    ).to(device)
if load_encoder_model_path != 'None':
    model_encoder.load_state_dict(torch.load(load_encoder_model_path))
    model_encoder.eval();
    
# ----------------------- define model projector ----------------------- #

if len(hidden_layers_projector) != 0:
    model_projector = cl_inference.nn_tools.define_MLP_model(
        hidden_layers_projector+[output_projector], output_encoder, bn=True
    ).to(device)
    if load_projector_model_path != 'None':
        model_projector.load_state_dict(torch.load(load_projector_model_path))
        model_projector.eval();
else:
    model_projector=None

# ----------------------- define model inference ----------------------- #

if len(hidden_layers_inference) != 0:
    if inference_loss == "MSE":
        output_dim_inference = NN_params_out
    else:
        n_tril = int(NN_params_out * (NN_params_out + 1) / 2)  # Number of parameters in lower triangular matrix, for symmetric matrix
        output_dim_inference = NN_params_out + n_tril  # Dummy output of neural network

    model_inference = cl_inference.nn_tools.define_MLP_model(
        hidden_layers_inference+[output_dim_inference], output_encoder, bn=True
    ).to(device)        
    if load_inference_model_path != 'None':
        model_inference.load_state_dict(torch.load(load_inference_model_path))
        model_inference.eval();
else:
    model_inference = None

# TRAIN

In [7]:
NN_epochs            = 800
NN_batches_per_epoch = 512
batch_size           = 256
lr                   = 0.0016300000000000002
weight_decay         = 0.
clip_grad_norm       = 45.54131781576748
seed_mode            = "random"
seed                 = 0

In [8]:
assert train_mode in ["train_CL", "train_inference_from_latents", "train_inference_fully_supervised", "train_CL_and_inference"], "mode must belong to one of the following categories: 'train_CL', 'train_inference_from_latents', 'train_inference_fully_supervised', or 'train_CL_and_inference'"

if train_mode == "train_CL":
    assert model_projector!=None, "model_projector must be provided"
    optimizer = torch.optim.AdamW([*model_encoder.parameters(), *model_projector.parameters()], lr=lr, betas=(0.9, 0.999), eps=1e-8, amsgrad=False, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=20, threshold=0.01, threshold_mode='abs', factor=0.3, min_lr=1e-8, verbose=True)
if train_mode == "train_inference_from_latents":
    assert model_inference!=None, "model_inference must be provided"
    optimizer = torch.optim.AdamW([*model_inference.parameters()], lr=lr, betas=(0.9, 0.999), eps=1e-8, amsgrad=False, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=30, threshold=0.15, threshold_mode='abs', factor=0.3, min_lr=1e-8, verbose=True)
if train_mode == "train_inference_fully_supervised":
    assert model_inference!=None, "model_inference must be provided"
    optimizer = torch.optim.AdamW([*model_encoder.parameters(), *model_inference.parameters()], lr=lr, betas=(0.9, 0.999), eps=1e-6, amsgrad=False, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=30, threshold=0.15, threshold_mode='abs', factor=0.3, min_lr=1e-8, verbose=True)
if train_mode == "train_CL_and_inference":
    assert model_projector!=None, "model_projector must be provided"
    assert model_inference!=None, "model_inference must be provided"
    optimizer = torch.optim.AdamW([*model_encoder.parameters(), *model_encoder.parameters(), *model_inference.parameters()], lr=lr, betas=(0.9, 0.999), eps=1e-6, amsgrad=False, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=30, threshold=0.15, threshold_mode='abs', factor=0.3, min_lr=1e-8, verbose=True)

assert seed_mode in ["random", "deterministic", "overfit"], " must belong to one of the following categories: 'random', 'deterministic' or 'overfit'"

if next(model_encoder.parameters()).is_cuda: device = "cuda"
else: device = "cpu"

/dipc/dlopez/.conda/envs/VE_torch/lib/python3.9/site-packages/torch/_compile.py:24: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return torch._dynamo.disable(fn, recursive)(*args, **kwargs)


In [9]:
def vector_to_Cov(vec, device="cuda"):
    """ Convert unconstrained vector into a positive-diagonal, symmetric covariance matrix
        by converting to cholesky matrix, then doing Cov = L @ L^T 
        (https://en.wikipedia.org/wiki/Cholesky_decomposition)
    """
    D = int((-1.0 + np.sqrt(1.0 + 8.0 * vec.shape[-1])) / 2.0)  # Infer dimensionality; D * (D + 1) / 2 = n_tril
    B = vec.shape[0]  # Batch dim
    
    # Get indices of lower-triangular matrix to fill
    tril_indices = torch.tril_indices(row=D, col=D, offset=0)
    
    # Fill lower-triangular Cholesky matrix
    L = torch.zeros((B, D, D)).to(device=device)
    mask1 = torch.zeros(L.shape, device=L.device, dtype=torch.bool)
    mask1[:, tril_indices[0], tril_indices[1]] = True
    L = L.masked_scatter(mask1, vec)
    
    # Enforce positive diagonals
    positive_diags = torch.nn.Softplus()(torch.diagonal(L, dim1=-1, dim2=-2))
    
    mask2 = torch.zeros(L.shape, device=L.device, dtype=torch.bool)
    mask2[:, range(L.shape[-1]), range(L.shape[-2])] = True
    L = L.masked_scatter(mask2, positive_diags)
    
    # Cov = L @ L^T 
    Cov = torch.einsum("bij, bkj ->bik",L, L)
    
    Cov = torch.from_numpy(np.array([0]))
    return Cov

In [10]:
min_val_loss = None
for tt in range(NN_epochs):
    print(f"\n\n\n-------------------------------------\n-------------- Epoch {tt+1} --------------\n-------------------------------------\n")

    if seed_mode == "deterministic": seed0 = NN_batches_per_epoch*tt
    else: seed0=0

    theta_true, xx, aug_params = dset(batch_size, seed=seed, to_torch=True, device=device)
    
    with torch.no_grad():

        len_batch = xx.shape[0]
        len_augs = xx.shape[1]
        len_batch_times_aug = len_batch * len_augs
        size_for_batch_resize = tuple([len_batch_times_aug,] + list(xx.shape[2:]))
        xx = torch.reshape(xx, size_for_batch_resize)

        theta_true = torch.repeat_interleave(theta_true, len_augs, axis=0)
        if aug_params is not None:
            aug_params = torch.reshape(aug_params, (len_batch_times_aug, aug_params.shape[-1]))
            theta_true = torch.concatenate((theta_true, aug_params), axis=-1)

        hh = model_encoder(xx.contiguous())

        zz = {}
        zz = model_projector(hh.contiguous())
        zz = torch.reshape(zz, tuple([len_batch, len_augs,] + list(zz.shape[1:])))

        yy = model_inference(hh.contiguous())
        theta_pred = yy[:, :theta_true.shape[-1]]
        Cov = vector_to_Cov(yy[:, theta_true.shape[-1]:]).to(device=device)




-------------------------------------
-------------- Epoch 1 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 2 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 3 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 4 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 5 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 6 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 7 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 8 --------------
-------------------------------------




-------------------------------------
-------------- 




-------------------------------------
-------------- Epoch 76 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 77 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 78 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 79 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 80 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 81 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 82 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 83 --------------
-------------------------------------




-------------------------------------
-------




-------------------------------------
-------------- Epoch 154 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 155 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 156 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 157 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 158 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 159 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 160 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 161 --------------
-------------------------------------




-------------------------------------




-------------------------------------
-------------- Epoch 229 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 230 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 231 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 232 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 233 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 234 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 235 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 236 --------------
-------------------------------------




-------------------------------------




-------------------------------------
-------------- Epoch 304 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 305 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 306 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 307 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 308 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 309 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 310 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 311 --------------
-------------------------------------




-------------------------------------

...




-------------------------------------
-------------- Epoch 450 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 451 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 452 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 453 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 454 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 455 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 456 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 457 --------------
-------------------------------------




-------------------------------------




-------------------------------------
-------------- Epoch 526 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 527 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 528 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 529 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 530 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 531 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 532 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 533 --------------
-------------------------------------




-------------------------------------




-------------------------------------
-------------- Epoch 602 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 603 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 604 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 605 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 606 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 607 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 608 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 609 --------------
-------------------------------------




-------------------------------------




-------------------------------------
-------------- Epoch 674 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 675 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 676 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 677 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 678 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 679 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 680 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 681 --------------
-------------------------------------




-------------------------------------




-------------------------------------
-------------- Epoch 747 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 748 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 749 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 750 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 751 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 752 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 753 --------------
-------------------------------------




-------------------------------------
-------------- Epoch 754 --------------
-------------------------------------




-------------------------------------

In [11]:
import pytorch

torch.set_num_threads(N_threads)
torch.set_num_interop_threads(N_threads)

for ii in range(10000):
    torch.zeros((512, 12, 12)).to(device="cuda")